# Lab 14: The 5 Universal API Components

## Naming what you've already been doing

Labs 9-13 all called Groq's chat completions endpoint through `call_groq`. This lab names the shape of that call - endpoint, method, headers, body, response - and wraps it into `SimpleAPIClient`, a small reusable class that isn't specific to Groq or to chat completions at all.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


## Exercise

Implement `SimpleAPIClient`: it stores a `base_url` and default `headers`, and its `.post(path, json_body)` method builds the full URL, sends the request, raises on a non-2xx response, and returns the parsed JSON body.

In [ ]:
class SimpleAPIClient:
    def __init__(self, base_url: str, headers: dict | None = None):
        self.base_url = base_url.rstrip("/")
        self.headers = headers or {}

    def post(self, path: str, json_body: dict) -> dict:
        """TODO: build the full endpoint URL (self.base_url + path), send a
        POST request with self.headers and json_body via `requests`, call
        response.raise_for_status() so a non-2xx status raises, and return
        response.json()."""
        raise NotImplementedError("SimpleAPIClient.post() is not implemented yet - replace the TODO with your code")


client = SimpleAPIClient(base_url="https://httpbin.org", headers={"X-Test": "1"})
assert client.base_url == "https://httpbin.org", 'client.base_url should be ' + repr("https://httpbin.org") + ", got " + repr(client.base_url)
result = client.post("/post", {"hello": "world"})
assert result["json"] == {"hello": "world"}, 'result["json"] should be ' + repr({"hello": "world"}) + ", got " + repr(result["json"])
assert result["headers"]["X-Test"] == "1", 'result["headers"]["X-Test"] should be ' + repr("1") + ", got " + repr(result["headers"]["X-Test"])

print("All checks passed (this made a real call to httpbin.org, a public echo service).")

## Solution

In [ ]:
class SimpleAPIClient:
    def __init__(self, base_url: str, headers: dict | None = None):
        self.base_url = base_url.rstrip("/")
        self.headers = headers or {}

    def post(self, path: str, json_body: dict) -> dict:
        url = f"{self.base_url}{path}"
        response = requests.post(url, headers=self.headers, json=json_body, timeout=30)
        response.raise_for_status()
        return response.json()


print(SimpleAPIClient("https://httpbin.org").post("/post", {"hello": "world"})["json"])

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Repeats lab 9's zero-shot sentiment call, but through `SimpleAPIClient` instead of the raw `requests.post` in `call_groq` - same 5 components, same result, different wrapping. Reuses `build_zero_shot_messages` / `parse_label` from lab 9 (redefined here so this notebook stays self-contained).

In [ ]:
def build_zero_shot_messages(review: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": "Classify the sentiment of the review as exactly one word: "
                       "'positive' or 'negative'. Reply with nothing else.",
        },
        {"role": "user", "content": review},
    ]


def parse_label(reply_text: str) -> str:
    normalized = reply_text.strip().strip(".").lower()
    if "negative" in normalized:
        return "negative"
    if "positive" in normalized:
        return "positive"
    return normalized


groq_client = SimpleAPIClient(
    base_url="https://api.groq.com/openai/v1",
    headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
)

review = "The battery life on this thing is incredible."
response_body = groq_client.post(
    "/chat/completions",
    {"model": DEFAULT_MODEL, "messages": build_zero_shot_messages(review), "temperature": 0.0},
)
reply_text = response_body["choices"][0]["message"]["content"]
print(f"Review: {review}")
print(f"Predicted sentiment: {parse_label(reply_text)}")